 # Notebook 4 - $\color{green}\text{Low Slope High Terrain \& Mapping}$

In [1]:
# Packages (geostack)
import numpy as np, pandas as pd, geopandas as gpd, rasterio as rio, matplotlib.pyplot as plt, xdem, seaborn as sns, warnings

from rasterio.plot import show
from rasterio.crs import CRS
from concurrent.futures import ThreadPoolExecutor
from pyproj import Proj, transform
warnings.filterwarnings("ignore")
from pathlib import Path

In [6]:
# User defined parameters. Run this cell each time you update one of the enclosed variables.
PROJ_TITLE = 'example_directory' # Create a succinct name with no spaces or leading digits to represent your project file for future exports.

PATH = Path.cwd() # Gets the user's current working directory to avoid the use of absolute paths.

***

## First we will identify points of Low Slope in High Terrain.  Defined as slope below 10th percentile for the watershed and terrain height above 90th percentile for the watershed.

In [7]:
def lsht(dem_dir=Path(PATH) / PROJ_TITLE / 'wshed_dems'):
    """
    Finds and assigns LSHT pixels within a watershed DEM.

    args: Path to dem

    returns: Dataframe with boolean values for LSHT and pixel coordinate values.
    """
    path_list = list(dem_dir.glob('*.tiff')) # Create a list of all the paths to our watershed DEMs.
    
    # Nested function to make a dataframe for storage.
    def df_maker(path):
        dem = xdem.DEM(path) # Load in the DEM.
        tri = dem.terrain_ruggedness_index(window_size=3) # Make an SLRM (Simple Local Relief Model) of the DEM.  3 pix window size.

        # Next we are going to find the 10th percentile relief for the dem.
        t_data = tri.data # Get our data.
        t_data = np.asarray(t_data)
        t_valid_mask = ~np.isnan(t_data)
        t_valid = t_data[t_valid_mask] # Create a mask that is false if it is nan.  Extract valid values.
        perc_10th = np.percentile(t_valid, 10) # From valid values, get 10th percentile.
        t_bool_mask = np.where(t_valid_mask & (t_data < perc_10th), True, False) # True if data is valid and below 10th percentile. False otherwise.

        # We now repeat the process to find the 90th percentile elevation.
        e_data = dem.data
        e_data = np.asarray(e_data)
        e_valid_mask = ~np.isnan(e_data)
        e_valid = e_data[e_valid_mask] # Create a mask that is false if it is nan.  Extract valid values.
        perc_90th = np.percentile(e_valid, 90) # From valid values, get 10th percentile.
        e_bool_mask = np.where(e_valid_mask &  (e_data > perc_90th), True, False)

        # Get the name for our export csv.
        name = path.name[:-5]

        # Create our df.
        # First get our coordinate values.
        nrows, ncols = tri.data.shape
        rows, cols = np.indices((nrows, ncols))
        xs, ys = tri.transform * (cols, rows)

        names = [name] * len(xs.flatten())

        data = {'lon':xs.flatten(), 'lat':ys.flatten(), 'b_10_s':t_bool_mask.flatten(), 'abv_90_t':e_bool_mask.flatten(), 'Wshed_name':names}
        df = pd.DataFrame(data=data)
        return df
        
    # Run our nested function in parallel for the watershed DEMs.
    with ThreadPoolExecutor() as executor:
        results_iter = executor.map(df_maker, path_list)
        results = list(results_iter)

    return results

# Run our Function
lsht_list = lsht()

## Then we retrieve all knickpoints from our directory, creates chiplots for each wshed.

In [8]:
def kp_retriever(lsht_list, chi_plot = False):
    '''
    Uses our LSHT dfs created in the above function and stored in some variable to identify knickpoints that border the LSHT terrain.  
    For information on why this matters, please consult the paper listed in notebook 1.

    args: lsht_list: a list of dataframes containing boolean values of whether or not a pixel is classified as LSHT.  chi_plot: tells the function to create chiplots.

    returns: List of dataframes contianing only the knickpoints that border our LSHT terrain.
    '''

    kp_list = [] # Create empty list
    if chi_plot == True: # Creates a directory to store chiplots if user requested.
        try:
            chiplots_dir = Path(PATH) / PROJ_TITLE / 'chi_plots'
            chiplots_dir.mkdir()
            print(f'Directory "{PATH}/{PROJ_TITLE}/chi_plots" created successfully.')
        except FileExistsError:
            print('Directory exists')
            chiplots_dir = Path(PATH) / PROJ_TITLE / 'chi_plots'
        except Exception as e:
            print(f'An error occurred: {e}')


    for df in lsht_list: # Iterates through dfs
        wshed_name = df.iloc[0].Wshed_name # Gets succint watershed name
        ksn_df = pd.read_csv(Path(PATH) / PROJ_TITLE / 'ksn_csvs' / f'{wshed_name}_reproj_ksn.csv') # Reads our K_{sn} dataframe for that watershed.  Contains steepness values.
        kp_df = pd.read_csv(Path(PATH) / PROJ_TITLE / 'kp_csvs' / f'{wshed_name}_reproj_knickpoints.csv') # Reads our knickpoint dataframe for that watershed.

        # Reprojecting
        in_proj = Proj(init='epsg:2283')
        out_proj = Proj(init='epsg:4269')
        easting = kp_df.x.values # Gets northing and easting for our transform function
        northing = kp_df.y.values
        longitude, latitude = transform(in_proj, out_proj, easting, northing) # Transfrorms.
        # Save our lat lon values to the df.
        kp_df['longitude'] = longitude
        kp_df['latitude'] = latitude
        kp_df['wshed'] = [wshed_name]*len(kp_df) # Create a column just containing the wshed name.

        if chi_plot == True: # Creates a chiplot if user requested.
            fig, ax = plt.subplots(figsize=(10,10))
            sns.scatterplot(data=ksn_df, x='flow_distance', y='chi', c='k', alpha=0.5, s=10, ax=ax)
            sns.scatterplot(data=ksn_df, x='flow_distance', y='chi', hue=np.abs(np.asarray(ksn_df['m_chi'])))
            plt.legend(title = r'$k_{sn}$')
            wshed_name_sp = wshed_name.replace('_', ' ')
            plt.title(f'{wshed_name_sp} $\chi$ plot')
            save_path = Path(str(chiplots_dir)) / f'{wshed_name}_chiplot.png'
            plt.savefig(save_path)
            plt.close()
        
        kp_list.append(kp_df) # Appends KP df to list.
    return kp_list

# Run our function
kp_list = kp_retriever(lsht_list, chi_plot=True)

Directory "d:\thesis_data_dict\notebooks_blank/example_directory/chi_plots" created successfully.


## Then, get only knickpoints within a specified distance of LSHT areas.  Also save plots of LSHT and knickpoints on a map.  Saves csvs of LSHT knickpoints.

In [9]:
# Makes a directory to store mapped LSHT kps.
try:
    kps_path = Path(PATH) / PROJ_TITLE / 'kps_mapped'
    kps_path.mkdir()
except FileExistsError:
    kps_path = Path(PATH) / PROJ_TITLE / 'kps_mapped'
except Exception as e:
    print(f'An Error occured : {e}')

try:
    lsht_kps_dir = Path(PATH) / PROJ_TITLE / 'lsht_kps'
    lsht_kps_dir.mkdir()
except FileExistsError:
    print('Directory exists!')

def mapper(df, ksn_df):
    # Parses out only LSHT pixels.
    ls = df['b_10_s'] == True
    ht = df['abv_90_t'] == True

    df_masked = df[ls & ht] # Masks dataframe by LSHT pixels.
    xs = df_masked.lon # Retrieves lat, lon.
    ys = df_masked.lat
    wshed_name = df_masked.iloc[0].Wshed_name # Retrieves wshed name

    fig, ax = plt.subplots(figsize=(20,20), dpi=200) # Creates figure object.

    with rio.open(Path(PATH)/ PROJ_TITLE / 'wshed_dems' / f'{wshed_name}.tiff') as src: # Plots the DEM as a basemap.
        show(src, cmap='terrain', ax=ax)

    ax.scatter(xs, ys, c='r', s=5) # Plots LSHT points in red.

    wshed_name_sp = wshed_name.replace('_', ' ') # Formatting title.
    title = f'{wshed_name_sp} LSHT plotted.'

    plt.title(label = title)

    # Makes a df of kps bordering LSHT points
    high_kps = gpd.sjoin_nearest(gpd.GeoDataFrame(ksn_df, geometry=gpd.points_from_xy(ksn_df['longitude'], ksn_df['latitude'], crs = CRS.from_epsg(2283))), 
                                 gpd.GeoDataFrame(df_masked, geometry=gpd.points_from_xy(df_masked['lon'], df_masked['lat']), crs=CRS.from_epsg(2283)), 
                                 distance_col = 'dist_from_lsht')

    LSHT_save_path = Path(str(lsht_kps_dir)) / f'{wshed_name}_lsht_kps.csv'
    pd.DataFrame(high_kps[high_kps.dist_from_lsht <= .05]).to_csv(LSHT_save_path) # Stores KP DF.

    # Plotting LSHT KPs on map with basemap and LSHT points.  Saves to directory.
    sns.scatterplot(data=high_kps[high_kps.dist_from_lsht <= .05], x='longitude', y='latitude', hue=np.abs(np.asarray(high_kps[high_kps.dist_from_lsht <= .05]['delta_ksn'])), ax=ax, palette='Greys', marker='s')
    ax.legend(title= r'Knickpoint $\Delta k_{sn}$)')
    save_path = Path(str(kps_path)) / f'{wshed_name}_mapped.png'
    plt.savefig(save_path)
    plt.close()

# runs our function
for lsht, kp in zip(lsht_list, kp_list):
    mapper(lsht, kp)